<a href="https://colab.research.google.com/github/ericyoc/the_applied_ai_universe_coding_guide/blob/main/voice_ai_agent_vishing_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The Setup (Prerequisites)
Before running the code, you will need:

ElevenLabs API Key: Get it from your Profile Settings.

Google Gemini API Key: Get a free one from Google AI Studio.

A Voice ID: In ElevenLabs, go to the Voice Lab, clone a voice, and copy its "Voice ID."

In [ ]:
!pip install -q -U google-genai elevenlabs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.4.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.4.0 which is incompatible.


In [ ]:
import re
import time
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript
import IPython.display as ipd
from google.genai import Client
from elevenlabs.client import ElevenLabs
from google.colab import userdata, drive, output

# 1. MOUNT DRIVE & SETUP
drive.mount('/content/drive', force_remount=True)
SAVE_PATH = "/content/drive/MyDrive/vishing_transcript.txt"

GEMINI_KEY = userdata.get('GEMINI_API_KEY')
ELEVEN_KEY = userdata.get('ELEVENLABS_API_KEY')
VOICE_ID = "pNInz6obpgDQGcFmaJgB"

gemini_client = Client(api_key=GEMINI_KEY)
eleven_client = ElevenLabs(api_key=ELEVEN_KEY)

# 2. CHAT SESSION
chat_session = gemini_client.chats.create(
    model='gemini-2.5-flash',
    config={
        "system_instruction": (
            "You are a professional IT Security Specialist. This is a vishing simulation. "
            "1. Build trust quickly. 2. Urgently ask for Employee ID. "
            "3. Max 25 words per response. Vary your phrasing."
        )
    }
)

# 3. UI COMPONENTS
output_area = widgets.Output(layout={'border': '1px solid #444', 'height': '400px', 'overflow_y': 'scroll'})
audio_area = widgets.Output()
text_input = widgets.Text(placeholder='Type or click Record...', layout={'width': '60%'})
send_button = widgets.Button(description="Send", button_style='success', layout={'width': '15%'})
record_button = widgets.Button(description="🎤 Record", button_style='danger', layout={'width': '20%'})

transcript = [f"--- VISHING LOG: {time.ctime()} ---"]
is_processing = False

# 4. RESET BUTTON FUNCTION
def reset_ui_state():
    global is_processing
    is_processing = False
    record_button.description = "🎤 Record"
    record_button.button_style = 'danger'
    record_button.disabled = False

# 5. CORE PROCESSING
def process_interaction(message):
    global is_processing
    if not message:
        reset_ui_state()
        return

    is_processing = True
    transcript.append(f"User: {message}")
    with output_area: print(f"\n[You]: {message}")

    try:
        response = chat_session.send_message(message)
        ai_text = re.sub(r'[\(\[].*?[\)\]]', '', response.text).strip()
        transcript.append(f"IT Agent: {ai_text}")

        with output_area: print(f" [IT Agent]: {ai_text}")

        audio_stream = eleven_client.text_to_speech.convert(
            text=ai_text, voice_id=VOICE_ID, model_id="eleven_flash_v2_5"
        )
        audio_bytes = b"".join(list(audio_stream))

        with audio_area:
            clear_output()
            display(ipd.Audio(audio_bytes, autoplay=True))
    except Exception as e:
        with output_area: print(f"⚠️ Error: {e}")

    reset_ui_state()

# 6. BROWSER STT WITH WATCHDOG
def start_browser_stt():
    js = Javascript("""
    (async () => {
        const recognition = new (window.SpeechRecognition || window.webkitSpeechRecognition)();
        recognition.lang = 'en-US';
        recognition.continuous = false;
        recognition.interimResults = false;

        // Watchdog: If no result after 6 seconds, force a reset
        const timeout = setTimeout(() => {
            recognition.stop();
            google.colab.kernel.invokeFunction('notebook.UpdateBtn', ['🎤 Record (Timeout)'], {});
        }, 6000);

        recognition.onstart = () => {
            google.colab.kernel.invokeFunction('notebook.UpdateBtn', ['🔴 LISTENING...'], {});
        };

        recognition.onresult = (event) => {
            clearTimeout(timeout);
            const text = event.results[0][0].transcript;
            google.colab.kernel.invokeFunction('notebook.ProcessVoice', [text], {});
        };

        recognition.onerror = (event) => {
            clearTimeout(timeout);
            google.colab.kernel.invokeFunction('notebook.UpdateBtn', ['❌ Mic Error'], {});
        };

        recognition.onend = () => {
            clearTimeout(timeout);
        };

        recognition.start();
    })();
    """)
    display(js)

# Callbacks
def update_button_callback(label):
    record_button.description = label
    if "Error" in label or "Timeout" in label:
        time.sleep(1)
        reset_ui_state()
    else:
        record_button.button_style = 'warning'

def process_voice_callback(text):
    process_interaction(text)

output.register_callback('notebook.ProcessVoice', process_voice_callback)
output.register_callback('notebook.UpdateBtn', update_button_callback)

# 7. HANDLERS
def on_record_click(b):
    record_button.disabled = True
    start_browser_stt()

def on_send_click(b):
    msg = text_input.value.strip()
    if msg.lower() in ['exit', 'quit', 'stop']:
        with open(SAVE_PATH, "w") as f: f.write("\n".join(transcript))
        text_input.close(); send_button.close(); record_button.close()
        with output_area: print(f"\n✅ SIMULATION ENDED. Saved to Drive.")
    else:
        text_input.value = ""
        process_interaction(msg)

send_button.on_click(on_send_click)
text_input.on_submit(on_send_click)
record_button.on_click(on_record_click)

# 8. LAUNCH
clear_output()
print("--- 🛡️ LIVE VOICE ATTACK SIMULATION ---")
display(output_area, widgets.HBox([text_input, send_button, record_button]), audio_area)

with output_area:
    print("System: Watchdog Timer Enabled. Button resets if no speech detected.")

--- 🛡️ LIVE VOICE ATTACK SIMULATION ---


Output(layout=Layout(border='1px solid #444', height='400px', overflow_y='scroll'))

Output()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import nbformat
from google.colab import _message
import json
from google.colab import files

# 1. Grab the notebook JSON directly from the current Colab session
raw_notebook = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
nb_json = raw_notebook['ipynb']

# 2. Fix the Widget Metadata
# Colab sometimes saves widgets without the 'state' key, which triggers the error
if 'metadata' in nb_json and 'widgets' in nb_json['metadata']:
    key = 'application/vnd.jupyter.widget-state+json'
    if key in nb_json['metadata']['widgets']:
        if 'state' not in nb_json['metadata']['widgets'][key]:
            nb_json['metadata']['widgets'][key]['state'] = {}
            print("✅ Fixed: Missing 'state' key added to widget metadata.")
    else:
        # If it's empty or weird, just remove it to be safe
        del nb_json['metadata']['widgets']
        print("✅ Fixed: Malformed widgets metadata removed.")
else:
    print("ℹ️ No widget issues found in current session metadata.")

# 3. Save the fixed version to a new file
fixed_filename = 'Fixed_Vishing_Lab.ipynb'
with open(fixed_filename, 'w', encoding='utf-8') as f:
    json.dump(nb_json, f)

# 4. Trigger the browser to download the clean file
print(f"🚀 Download starting: {fixed_filename}")
files.download(fixed_filename)

ℹ️ No widget issues found in current session metadata.
🚀 Download starting: Fixed_Vishing_Lab.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>